# Comparateur de prix SNCF Connect

Brest → Bourg-en-Bresse (gare réservable la plus proche de Saint-Claude) — aller 10/11/12 juil. 2026, retour 13/14.

> **Prérequis testés (2026-06-16) pour passer DataDome :**
> 1. **IP résidentielle/mobile** (partage de connexion 4G) — une IP datacenter/institutionnelle est bloquée d'office.
> 2. **Navigateur visible** (`HEADLESS=False`) via le display WSLg `:0` — le module le configure tout seul.
> 3. **Profil persistant** (`./browser_profile`) : garde le cookie DataDome. **La 1re fois**, un slider captcha peut s'afficher dans la fenêtre Chromium (visible sur le bureau Windows) → le résoudre à la main une fois, ensuite c'est automatique.
>
> **Saint-Claude est « Non réservable »** sur SNCF Connect (dernier tronçon TER non vendu en billet direct) → on prend **Bourg-en-Bresse** comme gare d'arrivée. Les prix viennent de l'API interne `bff/api/v1/itineraries`.

In [ ]:
import asyncio
import pandas as pd
from sncf_scraper import compare_dates, search_journeys

pd.set_option('display.max_rows', 200)

HEADLESS = False       # DataDome bloque le headless -> navigateur visible (WSLg :0, auto-configuré)
CARTE_JEUNE = True     # applique la Carte Avantage Jeune (plafonne les prix)
ORIGINE = 'Brest'
DESTINATION = 'Bourg-en-Bresse'   # gare réservable la + proche de Saint-Claude (non réservable)

DATES_ALLER  = ['2026-07-10', '2026-07-11', '2026-07-12']
DATES_RETOUR = ['2026-07-13', '2026-07-14']

## 1. Aller — Brest → Bourg-en-Bresse

In [ ]:
# max_pages=3 -> capte toute la journée (pagination "trajets suivants")
rows_aller = await compare_dates(ORIGINE, DESTINATION, DATES_ALLER,
                                 headless=HEADLESS, carte_jeune=CARTE_JEUNE, max_pages=3)
df_aller = pd.DataFrame(rows_aller)
df_aller

## 2. Retour — Bourg-en-Bresse → Brest

In [ ]:
rows_retour = await compare_dates(DESTINATION, ORIGINE, DATES_RETOUR,
                                  headless=HEADLESS, carte_jeune=CARTE_JEUNE, max_pages=3)
df_retour = pd.DataFrame(rows_retour)
df_retour

## 3. Tableau récapitulatif propre

In [ ]:
COLS = {'date': 'Date', 'depart': 'Départ', 'arrivee': 'Arrivée', 'duree': 'Durée',
        'correspondances': 'Corresp.', 'prix_eur': 'Prix (€)', 'transporteurs': 'Train'}

def mise_en_forme(df):
    if df.empty:
        print('Aucun résultat — vérifier raw/*.png (captcha ?) ou repasser en HEADLESS=False')
        return df
    d = df.rename(columns=COLS)
    cols = [c for c in COLS.values() if c in d.columns]
    return d[cols].sort_values(['Date', 'Départ']).reset_index(drop=True)

recap_aller = mise_en_forme(df_aller)
recap_aller

In [ ]:
recap_retour = mise_en_forme(df_retour)
recap_retour

## 4. Meilleur prix par date + meilleure combinaison aller/retour

In [ ]:
def meilleurs_par_date(df):
    if df.empty or 'prix_eur' not in df:
        return pd.DataFrame()
    d = df.dropna(subset=['prix_eur'])
    return d.loc[d.groupby('date')['prix_eur'].idxmin()].sort_values('date')

print('Aller — meilleur prix / date :')
display(meilleurs_par_date(df_aller))
print('Retour — meilleur prix / date :')
display(meilleurs_par_date(df_retour))

In [ ]:
# Meilleure combinaison aller + retour (prix mini)
ba = meilleurs_par_date(df_aller)
br = meilleurs_par_date(df_retour)
if not ba.empty and not br.empty:
    best_a = ba.loc[ba['prix_eur'].idxmin()]
    best_r = br.loc[br['prix_eur'].idxmin()]
    total = best_a['prix_eur'] + best_r['prix_eur']
    print(f"ALLER  {best_a['date']}  {best_a['depart']}→{best_a['arrivee']}  {best_a['prix_eur']:.2f} €")
    print(f"RETOUR {best_r['date']}  {best_r['depart']}→{best_r['arrivee']}  {best_r['prix_eur']:.2f} €")
    print(f"TOTAL  {total:.2f} €")
else:
    print('Pas assez de données pour combiner.')

## 5. (option) Export CSV

In [ ]:
if not df_aller.empty:
    recap_aller.to_csv('resultats_aller.csv', index=False)
if not df_retour.empty:
    recap_retour.to_csv('resultats_retour.csv', index=False)
print('Export terminé.')

## 6. Trajets fractionnés — découverte automatique via graphe TGV

`auto_split_search` parcourt un **graphe des villes reliées en TGV** (`sncf_scraper.TGV_GRAPH`) pour
trouver seul les routes candidates (directe + via hubs : Paris, Rennes, Lyon, Dijon…), cherche tous
les segments **en parallèle**, puis les chaîne sous contrainte de correspondance et de durée totale.

> Constat (11/07) : la Carte Avantage Jeune plafonnant déjà chaque billet, le fractionnement
> n'apporte pas de gain (direct 112 € < via Paris 143,80 €). Plus utile **sans carte** ou sur des
> trajets où le tarif direct explose. Adapter le graphe dans `sncf_scraper.py` au besoin.

In [ ]:
from sncf_scraper import auto_split_search, candidate_routes

# Le graphe TGV (sncf_scraper.TGV_GRAPH) trouve seul les routes candidates (directe + via hubs).
# Inutile de coder les villes-étapes à la main.
splits = await auto_split_search('Brest', 'Bourg-en-Bresse', '2026-07-11',
                                 max_legs=3, max_routes=8,
                                 carte_jeune=CARTE_JEUNE, heure='06h',
                                 min_layover_min=40, max_layover_min=300,
                                 max_total_h=14, concurrency=5)
df_split = pd.DataFrame(splits)
df_split[['route', 'prix_total', 'depart', 'arrivee', 'duree_totale', 'correspondances', 'detail']] if not df_split.empty else 'aucune combinaison'